# PPG Heart Rate Estimation with S4

Train S4 model on PPG dataset for continuous heart rate estimation.

This notebook follows a standard regression workflow:
- **Task**: Predict heart rate (bpm) from PPG windows
- **Model**: S4-based sequence model
- **Training**: MSE loss with MAE tracking
- **Evaluation**: Test set performance (MAE, RMSE)


In [ ]:
from typing import Dict, Any
from pathlib import Path
import torch

from src.datasets.ppg.ppg_config import PPGDaliaConfig

current_dir = Path.cwd()
project_root = current_dir.parent.parent.parent
data_root = str(project_root / "src" / "datasets" / "ppg" / "data")

## Task Definition

Import shared PPGTask from `src.tasks.ppg_task` to avoid code duplication.

In [ ]:
from src.notebooks.ppg.ppg_task import PPGTask

## Block Configuration Helper

Import S4 block config helper from `src.utils.block_configs` to avoid code duplication.

In [ ]:
from src.utils.block_configs import create_s4_block_cfg_ctor

## Configuration

**IMPORTANT**: Update `TRAIN_SUBJECTS`, `VAL_SUBJECT`, and `TEST_SUBJECT` to match your actual dataset subject IDs.

Place your PPG data in: `src/datasets/ppg/data/`

Expected structure:
```
data/
  S1/
    S1.pkl
  S2/
    S2.pkl
  ...
```

Each pickle file should contain:
- `signal['wrist']['BVP']`: PPG signal
- `label`: HR labels (bpm)


In [ ]:
TRAIN_SUBJECTS = ("S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8")
VAL_SUBJECT = "S9"
TEST_SUBJECT = "S10"

args: Dict[str, Any] = {
    # Data
    "data_root": data_root,
    "batch": 32,  # Reduced for MPS memory
    "data_loader_kwargs": {
        "num_workers": 0,
        "pin_memory": False,
        "persistent_workers": False,
        # PPGDaliaConfig parameters
        "subjects_train": TRAIN_SUBJECTS,
        "subject_val": VAL_SUBJECT,
        "subject_test": TEST_SUBJECT,
        "fs_in": 64.0,        # Input sampling rate (Hz)
        "fs": 100.0,          # Target sampling rate (Hz)
        "win_sec": 8,         # Window length (seconds)
        "stride_sec": 2,      # Stride (seconds)
        "do_bandpass": True,
        "low_hz": 0.5,
        "high_hz": 8.0,
    },

    # Training
    "epochs": 100,
    "lr": 5e-4,
    "wd": 1e-4,
    "amp": False,  # Disable AMP for MPS stability
    "save_dir": "./runs/ppg_s4_task",
    "warmup_epochs": 5,
    "patience": 5,
    "min_delta": 0.01,  # MAE improvement threshold

    # Model (reduced for MPS memory)
    "d_model": 128,      # Reduced from 256
    "depth": 4,          # Reduced from 6
    "dropout": 0.2,
    "mlp_ratio": 2.0,
    "droppath_final": 0.1,
    "layerscale_init": 1e-2,
    "residual_gain": 1.0,
    "pool": "mean",

    # S4-specific hyperparameters
    "d_state": 64,
    "mode": "s4d",           # 's4d', 's4', or 'diag'
    "bidirectional": False,  # Use bidirectional S4
}

args["block_cfg_ctor"] = create_s4_block_cfg_ctor(
    dropout=args["dropout"],
    mlp_ratio=args["mlp_ratio"],
    droppath_final=args["droppath_final"],
    layerscale_init=args["layerscale_init"],
    residual_gain=args["residual_gain"],
    pool=args["pool"],
    d_state=args["d_state"],
    mode=args["mode"],
    bidirectional=args["bidirectional"],
)

# Device selection
import os

if torch.backends.mps.is_available():
    args["device"] = torch.device("mps")
    # Set MPS memory management BEFORE any operations
    os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"
    torch.mps.set_per_process_memory_fraction(0.9)
    print("Using MPS (Apple Silicon) with reduced memory")
elif torch.cuda.is_available():
    args["device"] = torch.device("cuda")
    print("Using CUDA")
else:
    args["device"] = torch.device("cpu")
    args["amp"] = False
    print("Using CPU")

print(f"\nData root: {args['data_root']}")
print(f"Train subjects: {TRAIN_SUBJECTS}")
print(f"Val subject: {VAL_SUBJECT}")
print(f"Test subject: {TEST_SUBJECT}")
print(f"\nS4 Configuration:")
print(f"   d_state: {args['d_state']}")
print(f"   mode: {args['mode']}")
print(f"   bidirectional: {args['bidirectional']}")


## Training

In [ ]:
from src.train_utils.trainer import Trainer

# Define the task
task = PPGTask()

# Initialize trainer
trainer = Trainer(args=args, task=task)

# Train
best_metric, best_path = trainer.fit()

history = trainer.history

print(f"\n✅ Training complete! Best validation {trainer.early_key}: {best_metric:.4f}")
print(f"💾 Best model saved to: {best_path}")

## Test Evaluation

Import shared evaluation function from local `ppg_eval.py` to avoid code duplication.

In [ ]:
from ppg_eval import evaluate_best_model, prediction_visualization

preds, targets = evaluate_best_model(
    args=args,
    task=PPGTask(),
    best_model_path=args["save_dir"] + "/best.pt"
)

prediction_visualization(preds,targets)

In [ ]:
from src.utils.checkpoint import load_trainer_from_checkpoint
from src.utils.common import print_model_details

trainer = load_trainer_from_checkpoint(
        checkpoint_path=args["save_dir"] + "/best.pt",
        args=args,
        task=PPGTask(),
    )

print_model_details(model=trainer.model)

## Changes in datasets length

In [ ]:
from src.notebooks.ppg.ppg_eval import plot_train_history
from src.train_utils.trainer import Trainer
from ppg_eval import evaluate_best_model

for frac in [0.1, 0.25, 0.5]:

    print(f"\n🔄 Training with fraction: {frac}")

    args["fraction"] = frac
    args["save_dir"] = f"./runs/ppg_s4_task_frac_{int(frac*100)}"
    trainer = Trainer(args=args, task=PPGTask())
    best_metric, best_path = trainer.fit()

    print(f"\n✅ Training complete for fraction {frac}! Best validation {trainer.early_key}: {best_metric:.4f}")
    print(f"💾 Best model saved to: {best_path}")

    history = trainer.history

    plot_train_history(history)

    preds, targets = evaluate_best_model(
        args=args,
        task=PPGTask(),
        best_model_path=best_path
    )